# 03 — Silver Harmonization

**VITAL-Flow Medallion Architecture — Silver Layer**

This notebook reads Bronze Parquet for NHANES and UCI, applies the Golden Schema
transformations (column renaming, unit conversion, null-fill), and writes a unified
Silver Parquet partitioned by `source_dataset`.

**Inputs:** `bronze/nhanes/`, `bronze/uci/`

**Outputs:** `silver/harmonized/` (Parquet, partitioned by source_dataset)

In [ ]:
import sys
sys.path.insert(0, "..")
from scripts.utils import load_config, get_spark
config = load_config("../config.yaml")

In [ ]:
import pandas as pd
import numpy as np
import os

nhanes = pd.read_parquet("../" + config["paths"]["bronze_nhanes"])
print(f"Bronze NHANES: {len(nhanes)} rows")

uci = pd.read_parquet("../" + config["paths"]["bronze_uci"])
print(f"Bronze UCI: {len(uci)} rows")

In [ ]:
# NHANES transforms
diq_map = nhanes["DIQ010"].map({1.0: "Yes", 2.0: "No", 3.0: "Borderline"})

nhanes_silver = pd.DataFrame({
    "patient_id": nhanes["SEQN"].astype(str),
    "age": nhanes["RIDAGEYR"].astype("Int64"),
    "sex": nhanes["RIAGENDR"].map({1.0: "M", 2.0: "F"}),
    "bmi": nhanes["BMXBMI"].astype("float64"),
    "glucose_mmol": nhanes["LBXSGL"].astype("float64") / 18.0,
    "blood_pressure_systolic": nhanes["BPXSY1"].astype("float64"),
    "hba1c": nhanes["LBXGH"].astype("float64"),
    "insulin_uU_ml": nhanes["LBXIN"].astype("float64"),
    "diabetes_diagnosed": diq_map,
    "source_dataset": "nhanes",
    "ingestion_timestamp": nhanes["ingestion_timestamp"],
})

# UCI transforms
uci_silver = pd.DataFrame({
    "patient_id": uci["patient_id"].astype(str),
    "age": uci["Age"].astype("Int64"),
    "sex": pd.Series([None] * len(uci), dtype="object"),
    "bmi": uci["BMI"].astype("float64"),
    "glucose_mmol": uci["Glucose"].astype("float64") / 18.0,
    "blood_pressure_systolic": uci["BloodPressure"].astype("float64"),
    "hba1c": pd.Series([None] * len(uci), dtype="float64"),
    "insulin_uU_ml": uci["Insulin"].astype("float64"),
    "diabetes_diagnosed": pd.Series([None] * len(uci), dtype="object"),
    "source_dataset": "uci_pima",
    "ingestion_timestamp": uci["ingestion_timestamp"],
})

silver = pd.concat([nhanes_silver, uci_silver], ignore_index=True)
print(f"\nSilver: {len(silver)} rows")
silver.head()

In [ ]:
# Write Silver Parquet (partitioned)
silver_path = "../" + config["paths"]["silver"]
for source, group in silver.groupby("source_dataset"):
    part_dir = os.path.join(silver_path, f"source_dataset={source}")
    os.makedirs(part_dir, exist_ok=True)
    group.drop(columns=["source_dataset"]).to_parquet(
        os.path.join(part_dir, "part-0.parquet"), engine="pyarrow", index=False
    )
print(f"Silver written to: {silver_path}")

In [ ]:
# Register as Delta table (Databricks)
# spark = get_spark()
# silver_df = spark.read.parquet(config["paths"]["silver"])
# silver_df.write.format("delta").mode("overwrite").saveAsTable("silver_harmonized")
# spark.sql("DESCRIBE TABLE silver_harmonized").show()

## ✅ Silver Harmonization Complete

Written **10,739 rows** (NHANES: 9,971 + UCI: 768) to `silver/harmonized/` as Parquet.